In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
df = pd.read_excel("churn.xlsx", sheet_name="Customer Churn Data")

print("Dataset loaded successfully")
print(df.head())

Dataset loaded successfully
  Customer_ID  Gender Senior_Citizen  Tenure_Months   Contract_Type  \
0    TEL00001    Male             No           68.0  Month-to-month   
1    TEL00002    Male             No           64.0  Month-to-month   
2    TEL00003  Female            Yes           12.0  Month-to-month   
3    TEL00004    Male            Yes           51.0  Month-to-month   
4    TEL00005  Female             No           56.0  Month-to-month   

  Internet_Service Payment_Method  Monthly_Charges_INR  Total_Charges_INR  \
0              DSL  Bank Transfer               619.60           43796.41   
1            Fiber           Cash              1492.51           91446.97   
2            Fiber           Cash               974.56           12432.49   
3              DSL           Cash               309.56           16076.23   
4            Fiber           Cash              1410.78           77003.95   

  Tech_Support Online_Security  Support_Tickets_6M  Avg_Data_Usage_GB  \
0        

In [3]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nDataset Information:")
df.info()

print("\nStatistical Summary:")
print(df.describe())

Dataset Shape: (1200, 14)

Column Names:
Index(['Customer_ID', 'Gender', 'Senior_Citizen', 'Tenure_Months',
       'Contract_Type', 'Internet_Service', 'Payment_Method',
       'Monthly_Charges_INR', 'Total_Charges_INR', 'Tech_Support',
       'Online_Security', 'Support_Tickets_6M', 'Avg_Data_Usage_GB', 'Churn'],
      dtype='object')

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Customer_ID          1200 non-null   object 
 1   Gender               1165 non-null   object 
 2   Senior_Citizen       1200 non-null   object 
 3   Tenure_Months        1170 non-null   float64
 4   Contract_Type        1181 non-null   object 
 5   Internet_Service     1054 non-null   object 
 6   Payment_Method       1200 non-null   object 
 7   Monthly_Charges_INR  1165 non-null   float64
 8   Total_Charges_INR    1178 no

In [4]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
Customer_ID              0
Gender                  35
Senior_Citizen           0
Tenure_Months           30
Contract_Type           19
Internet_Service       146
Payment_Method           0
Monthly_Charges_INR     35
Total_Charges_INR       22
Tech_Support            36
Online_Security         32
Support_Tickets_6M      25
Avg_Data_Usage_GB       22
Churn                    0
dtype: int64


In [5]:
print("Invalid Tenure Values:")
print(df[(df["Tenure_Months"] < 0) |
         (df["Tenure_Months"] > 72)])

print("\nInvalid Monthly Charges:")
print(df[df["Monthly_Charges_INR"] < 0])

print("\nInvalid Total Charges:")
print(df[df["Total_Charges_INR"] < 0])

print("\nInvalid Support Tickets:")
print(df[df["Support_Tickets_6M"] < 0])

print("\nInvalid Data Usage:")
print(df[df["Avg_Data_Usage_GB"] < 0])

Invalid Tenure Values:
    Customer_ID  Gender Senior_Citizen  Tenure_Months   Contract_Type  \
102    TEL00103    Male             No           95.0  Month-to-month   
117    TEL00118  Female             No           80.0  Month-to-month   
515    TEL00516    Male             No           -3.0  Month-to-month   

    Internet_Service Payment_Method  Monthly_Charges_INR  Total_Charges_INR  \
102              DSL            UPI                  NaN           10973.95   
117            Fiber           Cash               930.34            3634.68   
515            Fiber           Cash                  NaN           21215.46   

    Tech_Support Online_Security  Support_Tickets_6M  Avg_Data_Usage_GB  \
102          Yes              No                 2.0              12.97   
117          Yes              No                 0.0                NaN   
515           No              No                 1.0             139.40   

         Churn  
102  Not Churn  
117  Not Churn  
515  Not Churn 

In [6]:
# Step 6: Handle Invalid Values

df.loc[(df["Tenure_Months"] < 0) |
       (df["Tenure_Months"] > 72), "Tenure_Months"] = np.nan

df.loc[df["Monthly_Charges_INR"] < 0,
       "Monthly_Charges_INR"] = np.nan

df.loc[df["Total_Charges_INR"] < 0,
       "Total_Charges_INR"] = np.nan

df.loc[df["Support_Tickets_6M"] < 0,
       "Support_Tickets_6M"] = np.nan

df.loc[df["Avg_Data_Usage_GB"] < 0,
       "Avg_Data_Usage_GB"] = np.nan

print("Invalid values handled successfully")

Invalid values handled successfully


In [7]:
# Step 7: Handle Missing Values

# Remove records where the target value is missing
df = df.dropna(subset=["Churn"])

# Identify numerical and categorical columns
numeric_cols = df.select_dtypes(include=np.number).columns

categorical_cols = df.select_dtypes(include="object").columns

# Fill numerical missing values using median
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values using mode
for col in categorical_cols:
    if col not in ["Customer_ID", "Churn"]:
        df[col] = df[col].fillna(df[col].mode()[0])

# Check remaining missing values
print("Missing values after preprocessing:")
print(df.isnull().sum())

Missing values after preprocessing:
Customer_ID            0
Gender                 0
Senior_Citizen         0
Tenure_Months          0
Contract_Type          0
Internet_Service       0
Payment_Method         0
Monthly_Charges_INR    0
Total_Charges_INR      0
Tech_Support           0
Online_Security        0
Support_Tickets_6M     0
Avg_Data_Usage_GB      0
Churn                  0
dtype: int64


In [8]:
# Step 8: Feature Engineering

# 1. Create Tenure Categories
df["Tenure_Category"] = pd.cut(
    df["Tenure_Months"],
    bins=[-1, 12, 24, 48, 72],
    labels=["New", "Medium", "Long", "Loyal"]
)

# 2. Calculate Average Monthly Spending
df["Average_Monthly_Spending"] = (
    df["Total_Charges_INR"] /
    df["Tenure_Months"].replace(0, np.nan)
)

df["Average_Monthly_Spending"] = (
    df["Average_Monthly_Spending"].fillna(0)
)

# 3. Create Service Usage Indicators
df["Internet_Usage"] = (
    df["Internet_Service"] != "No"
).astype(int)

df["Support_Usage"] = (
    df["Tech_Support"] == "Yes"
).astype(int)

df["Security_Usage"] = (
    df["Online_Security"] == "Yes"
).astype(int)

# 4. Calculate Total Services Used
df["Total_Services_Used"] = (
    df["Internet_Usage"] +
    df["Support_Usage"] +
    df["Security_Usage"]
)

print("Feature engineering completed successfully")

print(df[[
    "Tenure_Category",
    "Average_Monthly_Spending",
    "Internet_Usage",
    "Support_Usage",
    "Security_Usage",
    "Total_Services_Used"
]].head())

Feature engineering completed successfully
  Tenure_Category  Average_Monthly_Spending  Internet_Usage  Support_Usage  \
0           Loyal                644.064853               1              0   
1           Loyal               1428.858906               1              1   
2             New               1036.040833               1              0   
3           Loyal                315.220196               1              0   
4           Loyal               1375.070536               1              0   

   Security_Usage  Total_Services_Used  
0               0                    1  
1               0                    2  
2               1                    2  
3               0                    1  
4               1                    2  


In [9]:
# Step 9: Separate Features and Target

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Remove Customer_ID and Churn from input features
X = df.drop(["Customer_ID", "Churn"], axis=1)

# Convert target labels into numerical values
le = LabelEncoder()
y = le.fit_transform(df["Churn"])

# Split dataset into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training and testing split completed")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("Training records:", len(y_train))
print("Testing records:", len(y_test))

print("Target mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

Training and testing split completed
X_train shape: (960, 18)
X_test shape: (240, 18)
Training records: 960
Testing records: 240
Target mapping: {'Churn': np.int64(0), 'Not Churn': np.int64(1)}


In [10]:
# Step 10: Encode Categorical Features and Scale Numerical Features

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Identify numerical and categorical columns
numeric_features = X_train.select_dtypes(include=np.number).columns

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing methods
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Fit on training data and transform both datasets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Encoding and scaling completed successfully")
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Encoding and scaling completed successfully
Processed training shape: (960, 31)
Processed testing shape: (240, 31)


In [11]:
# Step 11: Train Two Classification Models

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Create the models
model1 = LogisticRegression(max_iter=1000, random_state=42)

model2 = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train Logistic Regression
model1.fit(X_train_processed, y_train)

# Train Random Forest
model2.fit(X_train_processed, y_train)

print("Both models trained successfully")

Both models trained successfully


In [12]:
# Step 12: Evaluate Model Performance

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Make predictions using both models
y_pred_lr = model1.predict(X_test_processed)
y_pred_rf = model2.predict(X_test_processed)

# Calculate evaluation metrics
results = []

for name, predictions in [
    ("Logistic Regression", y_pred_lr),
    ("Random Forest", y_pred_rf)
]:
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0)
    })

# Display comparison
results_df = pd.DataFrame(results)

print("Model Performance Comparison:")
print(results_df)

print("\nLogistic Regression Classification Report:")
print(classification_report(
    y_test, y_pred_lr,
    target_names=le.classes_,
    zero_division=0
))

print("\nRandom Forest Classification Report:")
print(classification_report(
    y_test, y_pred_rf,
    target_names=le.classes_,
    zero_division=0
))

Model Performance Comparison:
                 Model  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression  0.650000   0.679012  0.774648  0.723684
1        Random Forest  0.595833   0.638037  0.732394  0.681967

Logistic Regression Classification Report:
              precision    recall  f1-score   support

       Churn       0.59      0.47      0.52        98
   Not Churn       0.68      0.77      0.72       142

    accuracy                           0.65       240
   macro avg       0.63      0.62      0.62       240
weighted avg       0.64      0.65      0.64       240


Random Forest Classification Report:
              precision    recall  f1-score   support

       Churn       0.51      0.40      0.45        98
   Not Churn       0.64      0.73      0.68       142

    accuracy                           0.60       240
   macro avg       0.57      0.57      0.56       240
weighted avg       0.58      0.60      0.59       240



In [13]:
# Step 13: Select and Save the Best Model

import joblib

# Select the best model based on F1-score
best_model_name = results_df.loc[
    results_df["F1 Score"].idxmax(), "Model"
]

if best_model_name == "Logistic Regression":
    best_model = model1
else:
    best_model = model2

# Save the model and preprocessing objects
joblib.dump(best_model, "best_model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")
joblib.dump(le, "label_encoder.pkl")

print("Best Model:", best_model_name)
print("Model saved successfully")

Best Model: Logistic Regression
Model saved successfully


In [14]:
# Step 14: Install FastAPI Dependencies

%pip install fastapi uvicorn joblib pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [16]:
%%writefile app.py
from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import joblib

app = FastAPI(title="Customer Churn Prediction API")

model = joblib.load("best_model.pkl")
preprocessor = joblib.load("preprocessor.pkl")
label_encoder = joblib.load("label_encoder.pkl")


class CustomerData(BaseModel):
    Gender: str
    Senior_Citizen: int
    Tenure_Months: float
    Contract_Type: str
    Internet_Service: str
    Payment_Method: str
    Monthly_Charges_INR: float
    Total_Charges_INR: float
    Tech_Support: str
    Online_Security: str
    Support_Tickets_6M: float
    Avg_Data_Usage_GB: float


@app.get("/health")
def health():
    return {"status": "API is running"}


@app.post("/predict")
def predict(data: CustomerData):

    customer = pd.DataFrame([data.dict()])

    customer["Tenure_Category"] = pd.cut(
        customer["Tenure_Months"],
        bins=[-1, 12, 24, 48, 72],
        labels=["New", "Medium", "Long", "Loyal"]
    )

    customer["Average_Monthly_Spending"] = (
        customer["Total_Charges_INR"] /
        customer["Tenure_Months"].replace(0, float("nan"))
    ).fillna(0)

    customer["Internet_Usage"] = (
        customer["Internet_Service"] != "No"
    ).astype(int)

    customer["Support_Usage"] = (
        customer["Tech_Support"] == "Yes"
    ).astype(int)

    customer["Security_Usage"] = (
        customer["Online_Security"] == "Yes"
    ).astype(int)

    customer["Total_Services_Used"] = (
        customer["Internet_Usage"] +
        customer["Support_Usage"] +
        customer["Security_Usage"]
    )

    processed_data = preprocessor.transform(customer)
    prediction = model.predict(processed_data)
    result = label_encoder.inverse_transform(prediction)

    return {"Prediction": result[0]}

Writing app.py


In [17]:
# Step 16: Start FastAPI Server

import subprocess
import sys
import time
import requests
import os

# Start FastAPI server
log_file = open("uvicorn.log", "w")

server = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn",
        "app:app",
        "--host", "127.0.0.1",
        "--port", "8000"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(3)

# Check server status
if server.poll() is None:
    print("FastAPI server started successfully")
else:
    print("Server failed to start")
    log_file.close()
    with open("uvicorn.log", "r") as f:
        print(f.read())

FastAPI server started successfully


In [18]:
# Step 17: Test FastAPI Health Endpoint

response = requests.get("http://127.0.0.1:8000/health")

print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 200
Response: {'status': 'API is running'}


In [19]:
# Step 18: Test Customer Churn Prediction

customer_data = {
    "Gender": "Male",
    "Senior_Citizen": 0,
    "Tenure_Months": 12,
    "Contract_Type": "Month-to-Month",
    "Internet_Service": "Fiber Optic",
    "Payment_Method": "Electronic Check",
    "Monthly_Charges_INR": 1500,
    "Total_Charges_INR": 18000,
    "Tech_Support": "No",
    "Online_Security": "No",
    "Support_Tickets_6M": 5,
    "Avg_Data_Usage_GB": 25
}

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=customer_data
)

print("Status Code:", response.status_code)
print("Prediction:", response.json())

Status Code: 200
Prediction: {'Prediction': 'Churn'}


In [20]:
# Step 19: Install GitPython

%pip install GitPython

Note: you may need to restart the kernel to use updated packages.


In [21]:
# Step 20: Check Project Files

import os

print("Project Files:")

for file in os.listdir():
    print(file)

Project Files:
.ipynb_checkpoints
app.py
best_model.pkl
churn.xlsx
label_encoder.pkl
p.ipynb
preprocessor.pkl
Untitled.ipynb
uvicorn.log
__pycache__


In [22]:
# Step 21: Create Requirements File

import importlib.metadata as metadata

packages = [
    "fastapi",
    "uvicorn",
    "pandas",
    "numpy",
    "scikit-learn",
    "joblib",
    "pydantic",
    "requests"
]

with open("requirements.txt", "w") as f:
    for package in packages:
        version = metadata.version(package)
        f.write(f"{package}=={version}\n")

print("requirements.txt created successfully")

with open("requirements.txt", "r") as f:
    print(f.read())

requirements.txt created successfully
fastapi==0.141.1
uvicorn==0.52.1
pandas==2.3.3
numpy==2.3.5
scikit-learn==1.7.2
joblib==1.5.2
pydantic==2.12.4
requests==2.34.2



In [23]:
# Step 22: Create Git Ignore File

gitignore_content = """
__pycache__/
.ipynb_checkpoints/
*.pyc
uvicorn.log
churn.xlsx
.env
"""

with open(".gitignore", "w") as f:
    f.write(gitignore_content)

print(".gitignore created successfully")

.gitignore created successfully


In [24]:
# Step 23: Verify Final Project Files

for file in os.listdir():
    if file not in ["__pycache__", ".ipynb_checkpoints"]:
        print(file)

.gitignore
app.py
best_model.pkl
churn.xlsx
label_encoder.pkl
p.ipynb
preprocessor.pkl
requirements.txt
Untitled.ipynb
uvicorn.log


In [ ]:
# Step 25: Enter GitHub Credentials

import getpass

github_username = input("Enter your GitHub username: ")
github_token = getpass.getpass("Enter your GitHub token: ")

repo_name = "customer-churn-prediction"

print("GitHub details entered successfully")

In [25]:
# Step 26: Check Git Installation

import subprocess

result = subprocess.run(
    ["git", "--version"],
    capture_output=True,
    text=True
)

print(result.stdout if result.returncode == 0 else result.stderr)

git version 2.53.0.windows.3



In [26]:
# Step 27: Configure Git Username and Email

subprocess.run(["git", "config", "--global", "user.name",
                "Manikanta-palagani"])

subprocess.run(["git", "config", "--global", "user.email",
                "palaganimani5"])

print("Git configuration completed")

Git configuration completed
